# Validation of model predictions on Olga's reviews

This notebook isolates the events reviewed by Olga and compares the stored RemoteCLIP logistic-regression and XGBoost predictions with the current human event label. It uses database labels only; embeddings and model inference are not recomputed.

## Evaluation rules

- The cohort is selected explicitly by `forest_event_reviews.reviewer_name`.
- Human truth comes only from the label referenced by the current `human_label_id`.
- Only `VALID` events with `Logging` or `Wildfire` truth enter accuracy metrics.
- Rejected events are not model errors. They are summarized separately with their rejection reasons.
- The latest stored label per event and model source is used.
- Confidence intervals use grouped bootstrap resampling by point (`sample_id`), preserving all reviewed years of a sampled point.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "Scripts" / "ResearchTool").exists()
)
SCRIPTS_ROOT = PROJECT_ROOT / "Scripts"
if str(SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_ROOT))

from ResearchScripts.ml_tests.config import load_config
from ResearchScripts.ml_tests.review_validation import (
    run_reviewer_validation,
    validation_figures,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

## Parameters

Change `REVIEWER_NAME` only if future reviews are saved under another exact reviewer name. The analysis is read-only for SQLite and writes a timestamped report directory.

In [ ]:
REVIEWER_NAME = "Olga"
MODEL_SOURCES = ("REMOTECLIP_LOGREG", "XGBOOST")
BOOTSTRAP_RUNS = 2_000
RANDOM_SEED = 42
PROFILE = "quick"

config = load_config(profile=PROFILE)
database_path = config.path_value("database").resolve()
output_root = config.path_value("output_root").resolve() / "reviewer_validation"
database_path, output_root

## Run the database audit

This creates `REPORT.md`, CSV comparison tables, confusion matrices, and plots under the displayed run directory.

In [ ]:
engine = create_engine(f"sqlite:///{database_path.as_posix()}", future=True)
with Session(engine) as db:
    result = run_reviewer_validation(
        db,
        reviewer_name=REVIEWER_NAME,
        model_sources=MODEL_SOURCES,
        bootstrap_runs=BOOTSTRAP_RUNS,
        seed=RANDOM_SEED,
        output_root=output_root,
    )

print(f"Report directory: {result['run_dir']}")
display(pd.DataFrame([result["summary"]]).T.rename(columns={0: "value"}))

## Review coverage and class balance

These tables show what Olga has reviewed, which events are eligible for model scoring, and the human class distribution.

In [ ]:
display(pd.DataFrame(result["review_status_counts"]))
display(pd.DataFrame(result["human_label_counts"]))
display(pd.DataFrame(result["rejection_reason_counts"]))

## Metrics against accepted human labels

Aggregate metrics include prediction coverage. Per-class tables make the Logging/Wildfire imbalance explicit.

In [ ]:
model_metrics = pd.DataFrame(result["model_metrics"])
class_metrics = pd.DataFrame(result["class_metrics"])
display(model_metrics)
display(class_metrics)

## Confusion matrices

Rows are Olga's accepted labels; columns are model predictions.

In [ ]:
for model_name, rows in result["confusion_matrices"].items():
    display(Markdown(f"### {model_name}"))
    display(pd.DataFrame(rows))

figures = validation_figures(result)
for figure in figures.values():
    display(figure)

## Direct model comparison

The first table counts all pairs of stored predictions in Olga's reviewed cohort. The second table adds accepted human truth and separates both-correct, both-wrong, and one-model-only-correct outcomes.

In [ ]:
display(pd.DataFrame(result["model_agreement_counts"]))
display(pd.DataFrame(result["truth_model_combinations"]))
display(pd.DataFrame(result["joint_outcome_counts"]))

## Points explicitly reviewed by Olga

`reviewed_points.csv` contains one row per point, its reviewed event years, valid/rejected counts, human-label consistency, and model consistency across years.

In [ ]:
points = pd.DataFrame(result["sample_rows"])
display(points)

## Event-level audit table

Use this table to inspect individual years, notes, rejection reasons, prediction confidence, and the exact labels used in the comparison.

In [ ]:
events = pd.DataFrame(result["event_rows"])
event_columns = [
    "sample_id", "display_name", "event_id", "event_year",
    "event_loss_area_ha", "event_is_dominant", "review_status",
    "human_label", "rejection_reason", "review_notes",
    "remoteclip_label", "remoteclip_confidence",
    "xgboost_label", "xgboost_confidence",
    "models_agree", "joint_outcome", "review_updated_at",
]
display(events[event_columns])

## Generated report

The complete artifact set is safe to reuse without recomputing embeddings or inference. Rerunning this notebook creates a new timestamped directory so earlier reports remain unchanged.

In [ ]:
report_text = Path(result["report_path"]).read_text(encoding="utf-8")
display(Markdown(report_text))